# Preprocessing: Raw vs Processed

This cell loads raw and processed data for a panel and plots both over the same time range to compare preprocessing effects.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from spotanomaly2.infrastructure import storage

panel_id = "1"  # change as needed

# Find latest timestamped raw data directory
raw_base = Path("../data/raw")
raw_dirs = [d for d in raw_base.iterdir() if d.is_dir() and d.name.replace("_", "").isdigit()]
if not raw_dirs:
    raise ValueError("No timestamped raw data directories found")
latest_raw_dir = max(raw_dirs, key=lambda d: d.name)
print(f"Using raw data from: {latest_raw_dir.name}")

processed_dir = Path("../data/processed")

raw_df = storage.load_panel_parquet(latest_raw_dir, panel_id)
processed_df = storage.load_panel_parquet(processed_dir, panel_id)

# Use common columns for fair comparison
common_cols = [c for c in processed_df.columns if c in raw_df.columns]
if not common_cols:
    raise ValueError("No common columns found between raw and processed data")

# Align time range
start = max(raw_df.index.min(), processed_df.index.min())
end = min(raw_df.index.max(), processed_df.index.max())
raw_df = raw_df.loc[start:end, common_cols]
processed_df = processed_df.loc[start:end, common_cols]

# Plot
fig = make_subplots(rows=len(common_cols), cols=1, shared_xaxes=True, vertical_spacing=0.02,
                    subplot_titles=common_cols)

for i, col in enumerate(common_cols, start=1):
    fig.add_trace(
        go.Scatter(x=raw_df.index, y=raw_df[col], name=f"raw: {col}", line=dict(width=1), opacity=0.5),
        row=i, col=1
    )
    fig.add_trace(
        go.Scatter(x=processed_df.index, y=processed_df[col], name=f"processed: {col}", line=dict(width=1.5)),
        row=i, col=1
    )

fig.update_layout(
    height=max(400, 220 * len(common_cols)),
    title=f"Panel {panel_id}: Raw vs Processed (latest: {latest_raw_dir.name})",
    legend_title="Series",
    showlegend=True
)

fig.show()